# Anatomy of an App -- From Notebook to Dashboard

01-05 taught you Python, Plotly, pandas, pulling data from NOMAD, and the building blocks of a Voila UI. This notebook is the missing link: how those pieces become a real app that shows up as a card on the App Dashboard.

The worked example throughout is **Wetting Envelope** (`apps/Wetting_envelope/`) -- the smallest fully-unified app in this repo, small enough to read in full. Every code cell below opens the *actual* files on disk, so what you see is always current, not a snapshot that can drift out of date.


## 0. Running code cells

- Run a cell with **Shift + Enter**
- This notebook needs to run from inside the repo (`Learning/` sitting next to `apps/`) -- if a cell can't find a file, check your working directory.


In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent
APP_DIR = REPO_ROOT / "apps" / "Wetting_envelope"
assert APP_DIR.is_dir(), f"Can't find {APP_DIR} -- run this notebook from Learning/"

print("Repo root:", REPO_ROOT)


## 1. The four-file split, and why

Every unified app in `apps/<AppName>/` is split into exactly four Python files, each with one job:

| File | Job | May import `ipywidgets`? |
|---|---|---|
| `data_manager.py` | domain logic, data validation (often Pydantic models) | **No** |
| `plot_manager.py` | build Plotly figures from domain objects | **No** |
| `gui_components.py` | every widget, layout, and callback | Yes -- all of it lives here |
| `app.py` | thin assembly layer: wires the other three together | Yes, minimally |

The point of the split: `data_manager.py` and `plot_manager.py` can be unit tested with plain `pytest` -- no notebook, no browser, no widgets involved. Only `gui_components.py` and `app.py` need the interactive environment. Here's each file's own docstring making that promise explicit:


In [ ]:
for name in ["data_manager.py", "plot_manager.py", "gui_components.py", "app.py"]:
    text = (APP_DIR / name).read_text(encoding="utf-8")
    docstring = text.split('"""')[1].strip() if '"""' in text[:20] else "(no module docstring)"
    print(f"--- {name} ---")
    print(docstring)
    print()


## 2. The notebook is just two cells

Every app's `<app>.ipynb` has **exactly two cells**: one that logs notebook usage (and, for apps that need it, installs `hysprint_utils`), and one that builds and displays the app. Here's Wetting Envelope's actual notebook, cell by cell:


In [ ]:
import json

nb = json.loads((APP_DIR / "wetting_envelope.ipynb").read_text(encoding="utf-8"))
print(f"{len(nb['cells'])} cells\n")
for i, cell in enumerate(nb["cells"]):
    print(f"--- cell {i} ({cell['cell_type']}) ---")
    print("".join(cell["source"]))
    print()


That's the whole notebook -- all the real logic lives in the four `.py` files, not here. `%load_ext autoreload` / `%autoreload 2` is why editing `gui_components.py` and re-running cell 2 picks up your change without restarting the kernel.

Not every app's cell 1 looks this simple -- `App_dashboard`'s own notebook does an extra `pip install` + `sys.path` step first. That's a workaround for a one-time editable-install timing quirk, not something every app needs; if you ever hit an `ImportError` for `hysprint_utils` that only clears up after restarting the kernel, that's the issue, and the App_dashboard notebook is the reference fix.


## 3. Getting it onto the dashboard

The App Dashboard doesn't discover apps automatically -- each one is one `AppEntry` in `apps/App_dashboard/data_manager.py`'s `CATEGORIES` dict. Here's the real entry for Wetting Envelope:


In [ ]:
import re

dashboard_src = (REPO_ROOT / "apps" / "App_dashboard" / "data_manager.py").read_text(
    encoding="utf-8"
)
match = re.search(r'AppEntry\(\s*"Wetting_envelope".*?\),', dashboard_src, re.DOTALL)
print(match.group())


Field by field:

- `folder` -- the app's directory name under `apps/`
- `notebook` -- which `.ipynb` in that folder to render
- `name` / `description` / `icon` -- what shows on the card (icon is any [Font Awesome](https://fontawesome.com/search?o=r&m=free) class name)

Add an `AppEntry` like this one to the right category and your app has a card on the dashboard -- that's the entire registration step.


## 4. Packaging: `pyproject.toml`

Each app has its own `pyproject.toml`. The dependency line pointing at `shared/` is not boilerplate you can skip or reword -- it has to be this exact string for every app, since it's an absolute path into the NOMAD upload's own file layout:


In [ ]:
print((APP_DIR / "pyproject.toml").read_text(encoding="utf-8"))


## 5. Tests

`data_manager.py` and `plot_manager.py` having zero widget imports means they're importable and testable with plain `pytest` -- no notebook involved. Tests live at `tests/<app_name>/test_<app_name>.py` (never inside `apps/`), and each app's `conftest.py` loads its modules under a **unique** name so a full `pytest tests/` run doesn't collide with another app's same-named `data_manager` module:


In [ ]:
print((REPO_ROOT / "tests" / "wetting_envelope" / "conftest.py").read_text(encoding="utf-8"))


⚠️ **One repo-specific gotcha, worth knowing before it costs you an hour:** the repo root's own `secrets.py` (the local-testing token file) shadows Python's *stdlib* `secrets` module whenever the repo root ends up first on `sys.path` -- which breaks `numpy`/`pandas`/`plotly` imports with a confusing `ImportError: cannot import name 'randbits' from 'secrets'`. `tests/conftest.py` strips the repo root from `sys.path` specifically to prevent this. If that error ever resurfaces, check that file still exists before debugging anything else.


## 6. Landing the change: issues, PRs, versions

Condensed from `CONTRIBUTING.md` at the repo root (read it in full before your first PR):

1. **Open an issue first** on `nomad-hzb/nomad-pv-analysis-apps` -- bug fixes and features both start here. (Typo fixes and tooling-only changes are exempt.)
2. **Open a PR that references it**, e.g. `Fixes #123` in the description.
3. **Bump that app's version** in its own `pyproject.toml`, SemVer: patch for a fix, minor for a backward-compatible feature, major for a breaking change. Bump only the app you changed.
4. Releases are cut with `gh release create <tag> --generate-notes`, which builds notes from merged PR titles -- so a descriptive PR title *is* your changelog entry.


## Exercises (recommended)

1. Open `apps/bitmap_maker/` or `apps/Hansen_green_calculator/` and compare their file layout to Wetting Envelope's -- do they follow the same four-file split?
2. Sketch (in words, no need to write real code) what `data_manager.py` / `plot_manager.py` / `gui_components.py` would each contain for a tiny app that plots `y = x**n` for a user-chosen `n`.
3. Write the `AppEntry(...)` you'd add to `CATEGORIES` for that fictional app.
4. Find one function in `apps/Wetting_envelope/data_manager.py` and open `tests/wetting_envelope/test_wetting_envelope.py` to see how it's tested.
